# Piper TTS Training (Low Quality - 16kHz)
Use this notebook to train a **Piper** voice model from audio files generated by the Qwen-TTS generator.

### 🚀 Setup and Requirements
1. Ensure you are using an **A100 GPU** (Edit -> Notebook settings -> A100 GPU).
2. Your dataset should be in Google Drive (default: `/content/drive/MyDrive/piper_dataset`).

**Note:** Training is performed on the **local file system** to avoid Google Drive rate limits. Checkpoints are NOT automatically synced to Drive during training; you must copy them manually if you want to preserve them across sessions.

In [ ]:
# @title 0. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1. Install Dependencies (Patched for Python 3.12)
!apt-get update
!apt-get install -y espeak-ng libsndfile1-dev python3-dev

print("Cloning piper...")
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python

print("Installing python dependencies...")
!pip install piper-phonemize-cross
!pip install pytorch-lightning==1.9.0 onnxruntime onnx-simplifier Cython>=0.29.0 librosa pandas tqdm
!pip install -e . --no-deps

print("Building monotonic alignment...")
!bash build_monotonic_align.sh

In [ ]:
# @title 2. Prepare Dataset (Resample to 16kHz & Fix Metadata)
import os
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

input_dir = "/content/drive/MyDrive/piper_dataset" # @param {type:"string"}
output_dir = "/content/piper_training" # @param {type:"string"}
sample_rate = 16000

os.makedirs(f"{output_dir}/wav", exist_ok=True)

metadata_path = os.path.join(input_dir, "metadata.csv")
new_metadata_path = os.path.join(output_dir, "metadata.csv")

print("🚀 Processing audio files and metadata...")
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        lines = f.readlines()
    
    with open(new_metadata_path, "w") as f_out:
        for line in tqdm(lines):
            if "|" not in line: continue
            filename, text = line.strip().split("|", 1)
            
            file_id = filename.replace(".wav", "")
            f_out.write(f"{file_id}|{text}\n")
            
            src_path = os.path.join(input_dir, "wavs", filename)
            dst_path = os.path.join(output_dir, "wav", f"{file_id}.wav")
            
            if not os.path.exists(dst_path):
                y, sr = librosa.load(src_path, sr=None)
                y_resampled = librosa.resample(y, orig_sr=sr, target_sr=sample_rate)
                sf.write(dst_path, y_resampled, sample_rate)

    print(f"✅ Dataset prepared at {output_dir}")
else:
    print(f"❌ metadata.csv not found at {metadata_path}")

In [ ]:
# @title 3. Run Piper Preprocessing
language = "en-us" # @param {type:"string"}

!python3 -m piper_train.preprocess \
  --language {language} \
  --input-dir {output_dir} \
  --output-dir {output_dir}/preprocessed \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate {sample_rate}

In [ ]:
# @title 4. Download Pre-trained Checkpoint (Lessac Low)
checkpoint_url = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/low/epoch%3D2307-step%3D558536.ckpt"
checkpoint_path = "/content/lessac_low.ckpt"

if not os.path.exists(checkpoint_path):
    print("Downloading base checkpoint...")
    !wget -O {checkpoint_path} "{checkpoint_url}"
else:
    print("Checkpoint already exists.")

In [ ]:
# @title 5. Training (Fine-tuning)
import os
import shutil
import glob

# Set environment variable to allow loading legacy Piper checkpoints (Python 3.12/PyTorch 2.6 fix)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

batch_size = 64 # @param {type:"integer"}
max_epochs = 3000 # @param {type:"integer"}
checkpoint_epochs = 1 # @param {type:"integer"}
local_log_dir = "/content/piper_logs" # @param {type:"string"}
drive_backup_dir = "/content/drive/MyDrive/piper_training_logs" # @param {type:"string"}
base_checkpoint = "/content/lessac_low.ckpt" # @param {type:"string"}

os.makedirs(local_log_dir, exist_ok=True)
os.makedirs(drive_backup_dir, exist_ok=True)

# Find the latest checkpoint to resume from (Checking both Local and Drive)
ckpt_pattern_local = os.path.join(local_log_dir, "lightning_logs/version_*/checkpoints/*.ckpt")
ckpt_pattern_drive = os.path.join(drive_backup_dir, "lightning_logs/version_*/checkpoints/*.ckpt")

existing_ckpts = glob.glob(ckpt_pattern_local) + glob.glob(ckpt_pattern_drive)

if existing_ckpts:
    resume_checkpoint = max(existing_ckpts, key=os.path.getctime)
    print(f"🔄 Found existing checkpoint: {resume_checkpoint}")
    print("Resuming training from your latest progress...")
else:
    resume_checkpoint = base_checkpoint
    print(f"🌟 No existing checkpoints found. Starting fresh from {base_checkpoint}")

%cd {local_log_dir}

%reload_ext tensorboard
%tensorboard --logdir {local_log_dir}

print(f"🚀 Starting training. max_epochs is set to {max_epochs}.")
print(f"Note: The base checkpoint is at epoch 2307, so training will continue from there.")

!python3 -m piper_train \
    --dataset-dir {output_dir}/preprocessed \
    --default_root_dir {local_log_dir} \
    --accelerator 'gpu' \
    --devices 1 \
    --batch-size {batch_size} \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs {max_epochs} \
    --resume_from_checkpoint {resume_checkpoint} \
    --checkpoint-epochs {checkpoint_epochs} \
    --precision 32 \
    --quality medium \
    --hidden-channels 192 \
    --inter-channels 192 \
    --filter-channels 768 \
    --n-layers 6 \
    --n-heads 2 \
    --log_every_n_steps 1

# After training completes, offer to backup the latest checkpoint to Drive
print("\n--- Training Session Finished ---")
latest_local_ckpts = glob.glob(ckpt_pattern_local)
if latest_local_ckpts:
    latest_ckpt = max(latest_local_ckpts, key=os.path.getctime)
    print(f"Latest local checkpoint: {latest_ckpt}")
    # Note: We don't auto-copy everything to avoid rate limits during training,
    # but copying the final one is usually safe.
    # !cp {latest_ckpt} {drive_backup_dir}/


In [ ]:
# @title 6. Export to ONNX
import glob
import os
import re

# Look for checkpoints in both Local and Drive log directories
ckpt_pattern_local = os.path.join(local_log_dir, "lightning_logs/version_*/checkpoints/*.ckpt")
ckpt_pattern_drive = os.path.join(drive_backup_dir, "lightning_logs/version_*/checkpoints/*.ckpt")
ckpts = glob.glob(ckpt_pattern_local) + glob.glob(ckpt_pattern_drive)

if not ckpts:
    print("❌ No checkpoints found! Check Step 5.")
else:
    # Use the same smart ranking to find the absolute latest checkpoint
    def get_rank(path):
        v_match = re.search(r'version_(\d+)', path)
        e_match = re.search(r'epoch=(\d+)', path)
        v = int(v_match.group(1)) if v_match else 0
        e = int(e_match.group(1)) if e_match else 0
        return (v, e)
    
    last_ckpt = max(ckpts, key=get_rank)
    
    # Extract epoch for the filename
    epoch_match = re.search(r'epoch=(\d+)', last_ckpt)
    epoch_num = epoch_match.group(1) if epoch_match else "final"
    
    onnx_filename = f"rocky_model_{epoch_num}.onnx"
    onnx_path = os.path.join("/content/drive/MyDrive", onnx_filename)

    print(f"📦 Exporting Epoch {epoch_num}...")
    print(f"Source: {last_ckpt}")
    print(f"Target: {onnx_path}")
    
    !python3 -m piper_train.export_onnx {last_ckpt} {onnx_path}

    # Copy config and rename to match ONNX
    !cp {output_dir}/preprocessed/config.json {onnx_path}.json

    print(f"✅ SUCCESS! Download these files from Drive root:")
    print(f"1. {onnx_filename}")
    print(f"2. {onnx_filename}.json")
